# Monitoring a Deployed Model and Knowing When to Retrain

A model that was accurate last month may be inaccurate today. The real world changes — new user behaviour, seasonal patterns, data pipeline bugs — and models degrade silently unless you monitor them. This notebook builds a simple monitoring pipeline: log predictions, detect distribution shift, and decide when to trigger a retraining job.

**Learning objectives**
1. Explain the difference between data drift and concept drift.
2. Build a request logger that writes predictions to a JSON-lines file.
3. Simulate a distribution shift and visualize how model confidence drops.
4. Implement a simple statistical drift detector using mean/std thresholds.
5. Define a retraining trigger rule based on drift detection or accuracy degradation.


## 1  Why monitoring matters

Two types of drift cause silent model degradation:

- **Data drift**: the distribution of input features changes. Example: a product recommendation model trained on desktop users starts receiving mostly mobile traffic.
- **Concept drift**: the relationship between features and labels changes. Example: a fraud model trained before a new type of fraud becomes common.

Without logs, you cannot detect either. You need to record inputs and predictions for every request.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import json
import time
import os
from datetime import datetime, timedelta

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])
model.fit(X_train, y_train)

# Compute training distribution baseline
TRAIN_MEAN = X_train.mean(axis=0)
TRAIN_STD = X_train.std(axis=0)
CLASS_NAMES = list(iris.target_names)

print(f"Model accuracy on test set: {model.score(X_test, y_test):.2%}")
print(f"Training distribution mean (per feature): {TRAIN_MEAN.round(3)}")
print(f"Training distribution std  (per feature): {TRAIN_STD.round(3)}")


## 2  Request logger

Every prediction gets logged with its timestamp, raw features, prediction, and confidence. We write to a JSON-lines file (one JSON object per line) — simple, append-friendly, and parseable.


In [ ]:
LOG_PATH = "/tmp/prediction_log.jsonl"

# Clear any previous log
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)

def log_prediction(features, prediction, confidence, timestamp=None):
    """Append one prediction record to the JSON-lines log."""
    record = {
        "timestamp": (timestamp or datetime.utcnow()).isoformat(),
        "features": list(float(v) for v in features),
        "prediction": prediction,
        "confidence": round(float(confidence), 4)
    }
    with open(LOG_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")
    return record


def predict_and_log(model, features, timestamp=None):
    """Run prediction and log it. Returns the log record."""
    arr = np.array(features).reshape(1, -1)
    proba = model.predict_proba(arr)[0]
    class_idx = int(np.argmax(proba))
    return log_prediction(
        features,
        CLASS_NAMES[class_idx],
        proba[class_idx],
        timestamp
    )

# Quick test: log one prediction
sample = X_test[0]
record = predict_and_log(model, sample)
print("Sample log record:")
print(json.dumps(record, indent=2))


## 3  Simulate 200 predictions — normal then shifted distribution

We simulate a real deployment: the first 100 requests come from the normal Iris distribution; the next 100 come from a shifted distribution (features scaled up by 2x, as if new sensors were calibrated differently). We backdate the timestamps to simulate a 200-request session over time.


In [ ]:
np.random.seed(0)
base_time = datetime.utcnow() - timedelta(hours=4)

# First 100 requests — from the training distribution
for i in range(100):
    idx = np.random.randint(0, len(X_test))
    ts = base_time + timedelta(minutes=i * 2)
    predict_and_log(model, X_test[idx], timestamp=ts)

# Next 100 requests — shifted distribution (all features scaled up by 2x)
for i in range(100):
    idx = np.random.randint(0, len(X_test))
    shifted = X_test[idx] * 2.0  # simulated sensor recalibration / data drift
    ts = base_time + timedelta(minutes=(100 + i) * 2)
    predict_and_log(model, shifted, timestamp=ts)

# Count log entries
with open(LOG_PATH) as f:
    log_records = [json.loads(line) for line in f]

print(f"Total logged predictions: {len(log_records)}")
print(f"First timestamp : {log_records[0]['timestamp']}")
print(f"Last timestamp  : {log_records[-1]['timestamp']}")


## 4  Analyze the logs — confidence over time

If the distribution shifts, the model encounters inputs it wasn't trained for. Its confidence will drop because the class probabilities become more uncertain. Plotting confidence over time is a simple early warning signal.


In [ ]:
import matplotlib.pyplot as plt

confidences = [r["confidence"] for r in log_records]
indices = list(range(len(confidences)))

# Rolling average (window = 10)
window = 10
rolling_avg = [
    float(np.mean(confidences[max(0, i - window): i + 1]))
    for i in indices
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(indices[:100], confidences[:100], alpha=0.4, color="steelblue", label="Normal distribution", s=15)
ax.scatter(indices[100:], confidences[100:], alpha=0.4, color="tomato", label="Shifted distribution", s=15)
ax.plot(indices, rolling_avg, color="black", linewidth=1.5, label=f"Rolling avg (window={window})")
ax.axvline(x=100, color="orange", linestyle="--", linewidth=1.5, label="Distribution shift point")
ax.set_xlabel("Request number")
ax.set_ylabel("Model confidence")
ax.set_title("Model confidence over time — confidence drops after distribution shift")
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/confidence_over_time.png", dpi=90)
plt.show()

mean_normal = float(np.mean(confidences[:100]))
mean_shifted = float(np.mean(confidences[100:]))
print(f"Mean confidence (normal distribution)  : {mean_normal:.3f}")
print(f"Mean confidence (shifted distribution) : {mean_shifted:.3f}")
print(f"Drop in confidence: {mean_normal - mean_shifted:.3f}")


## 5  Simple drift detection — compare recent inputs to training baseline

For each feature, we compare the mean of recent requests to the training mean. If the difference exceeds 2 standard deviations (of the training distribution), we flag it as drift.


In [ ]:
FEATURE_NAMES = list(iris.feature_names)
DRIFT_THRESHOLD_STD = 2.0  # flag if recent mean deviates by more than 2 training std

def detect_drift(records, train_mean, train_std, threshold_std=2.0):
    """
    Compare the mean of recent records' features to the training distribution.
    Returns a dict with per-feature drift flags.
    """
    recent_features = np.array([r["features"] for r in records])
    recent_mean = recent_features.mean(axis=0)

    deviations = np.abs(recent_mean - train_mean) / (train_std + 1e-8)
    drifted = deviations > threshold_std

    result = {
        "any_drift": bool(np.any(drifted)),
        "features": {}
    }
    for i, fname in enumerate(FEATURE_NAMES):
        result["features"][fname] = {
            "train_mean": round(float(train_mean[i]), 4),
            "recent_mean": round(float(recent_mean[i]), 4),
            "deviation_std": round(float(deviations[i]), 3),
            "drifted": bool(drifted[i])
        }
    return result


# Check normal window (requests 0-99)
drift_normal = detect_drift(log_records[:100], TRAIN_MEAN, TRAIN_STD, DRIFT_THRESHOLD_STD)
print("Drift check on normal window (requests 0-99):")
print(f"  Any drift detected: {drift_normal['any_drift']}")

# Check shifted window (requests 100-199)
drift_shifted = detect_drift(log_records[100:], TRAIN_MEAN, TRAIN_STD, DRIFT_THRESHOLD_STD)
print("\nDrift check on shifted window (requests 100-199):")
print(f"  Any drift detected: {drift_shifted['any_drift']}")
print("  Per-feature breakdown:")
for fname, info in drift_shifted["features"].items():
    flag = "DRIFT" if info["drifted"] else "ok"
    print(f"    {fname:<25} train_mean={info['train_mean']:.3f}  "
          f"recent_mean={info['recent_mean']:.3f}  "
          f"deviation={info['deviation_std']:.1f}x std  [{flag}]")


## 6  Retraining trigger

Once drift is detected (or confidence drops below a threshold), you need a rule to decide whether to retrain. Here we implement a simple decision function.


In [ ]:
def should_retrain(records, train_mean, train_std, min_confidence=0.80, drift_threshold_std=2.0):
    """
    Returns True and a reason if retraining should be triggered.
    Two signals checked: (1) average confidence dropped, (2) feature drift detected.
    """
    if not records:
        return False, "No records to evaluate"

    # Signal 1: confidence degradation
    avg_conf = float(np.mean([r["confidence"] for r in records]))
    if avg_conf < min_confidence:
        return True, f"Average confidence {avg_conf:.3f} below threshold {min_confidence}"

    # Signal 2: feature drift
    drift = detect_drift(records, train_mean, train_std, drift_threshold_std)
    if drift["any_drift"]:
        drifted_features = [
            f for f, info in drift["features"].items() if info["drifted"]
        ]
        return True, f"Drift detected in features: {drifted_features}"

    return False, "No retraining needed"


# Evaluate on the normal window
retrain, reason = should_retrain(log_records[:100], TRAIN_MEAN, TRAIN_STD)
print(f"Normal window  -> Retrain: {retrain}  |  Reason: {reason}")

# Evaluate on the shifted window
retrain, reason = should_retrain(log_records[100:], TRAIN_MEAN, TRAIN_STD)
print(f"Shifted window -> Retrain: {retrain}  |  Reason: {reason}")

# Evaluate on the full log
retrain, reason = should_retrain(log_records, TRAIN_MEAN, TRAIN_STD)
print(f"Full log       -> Retrain: {retrain}  |  Reason: {reason}")


## Summary

The monitoring loop for a deployed model has four stages:

| Stage | What happens |
|---|---|
| Log | Every request writes timestamp, features, prediction, confidence to a log file |
| Analyze | Periodically compute rolling averages and compare feature means to training baseline |
| Alert | If confidence drops or drift exceeds threshold, flag for human review |
| Retrain | Collect new labeled data, retrain, validate, deploy — then the cycle restarts |

Data drift usually *does* show up as a confidence drop — the model sees inputs unlike anything in its training data. Concept drift is the dangerous one: the inputs can look completely normal (so confidence stays high) while the correct answer has changed underneath you. That is why concept drift generally cannot be caught by watching inputs or confidence alone — you need fresh labels to measure accuracy directly.


## Self-check

1. **What is the difference between data drift and concept drift?** Look at the Section 1 definitions and think of one real-world example of each from an industry you know.
2. **How would you know your model's accuracy has dropped if you don't have labels for new data?** Confidence is a proxy — but what are its limitations? When would high confidence still mean wrong predictions?
3. **Name two signals that should trigger a retraining pipeline.** Look at `should_retrain()` in Section 6 — what are the two signals it checks? Can you think of a third that would be more reliable than either?
